# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` per the STTM *Silver to Gold (CareSync Health Network)*.

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `<catalog>.silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`, `silver.calendar` on `date_key = yyyyMMdd(test_date)`
* **Window:** `LAG(total_tests)` partitioned by `hospital_id` ordered by `summary_date` (previous day with activity)
* **Filters:** none
* **Load type:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "healthcare_databricks", "Catalog")
catalog = dbutils.widgets.get("catalog")

target_table = f"{catalog}.gold.hospital_daily_summary"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend (STTM silver to gold)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                               AS summary_date,
    COUNT(*)                                                AS total_tests,
    COUNT(DISTINCT patient_id)                              AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                               AS active_doctor_count,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {catalog}.silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(t.hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                             AS hospital_name,
  CAST(t.summary_date AS DATE)                                AS summary_date,
  CAST(c.day_name AS STRING)                                  AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                               AS is_weekend,
  CAST(t.total_tests AS INT)                                  AS total_tests,
  CAST(t.unique_patients_tested AS INT)                       AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                          AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                          AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                          AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / NULLIF(t.total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / NULLIF(t.total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                         AS prev_day_total_tests,
  CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0 / NULLIF(t.prev_day_total_tests, 0), 2) AS DECIMAL(6,2)) AS test_volume_change_pct
FROM trend t
LEFT JOIN {catalog}.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
# Grain check: (hospital_id, summary_date) must be unique; total tests must reconcile to silver.lab_results
display(spark.sql(f"""
SELECT
  COUNT(*)                                                      AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)                     AS distinct_hospital_days,
  SUM(total_tests)                                              AS sum_total_tests,
  (SELECT COUNT(*) FROM {catalog}.silver.lab_results)           AS silver_lab_results,
  SUM(CASE WHEN day_name IS NULL THEN 1 ELSE 0 END)             AS missing_calendar_join,
  SUM(CASE WHEN hospital_name IS NULL THEN 1 ELSE 0 END)        AS missing_hospital_name,
  SUM(CASE WHEN prev_day_total_tests IS NULL THEN 1 ELSE 0 END) AS first_day_rows,
  COUNT(DISTINCT hospital_id)                                   AS hospitals
FROM {target_table}
"""))
display(spark.table(target_table).orderBy("hospital_id", "summary_date").limit(10))

In [0]:
# Return the gold table record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table(target_table)

record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))